# ClassicModels - Dashboard Analitico (Task 3)

Este notebook consulta o star schema materializado pela Task 2 (fact_orders, dim_customers, dim_products, dim_dates, dim_countries) atraves do Amazon Athena, usando o catalogo de tabelas registrado no AWS Glue Data Catalog pela Task 3.

Funciona em dois modos:

- Local (Jupyter Lab / VS Code): le configuracao do .env ou diretamente dos outputs do Terraform.
- AWS SageMaker Notebook Instance: o lifecycle hook do SageMaker injeta as variaveis no arquivo task3/.env antes do kernel iniciar.

Todas as queries vao para o mesmo workgroup do Athena, que controla o local de resultados em S3.


## Setup e configuracao

A configuracao e resolvida nessa ordem, com fallback:
1. variaveis de ambiente ja definidas no kernel;
2. arquivo .env no diretorio do notebook (gerado pelo SageMaker lifecycle ou criado manualmente em dev local);
3. arquivo ../.env (raiz do projeto);
4. outputs do Terraform via terraform output -json.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import awswrangler as wr
import boto3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from ipywidgets import (
    Dropdown,
    IntSlider,
    Output,
    SelectionRangeSlider,
    VBox,
    HBox,
    interactive_output,
)

sns.set_theme(style="whitegrid")


In [ ]:
def _try_load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


def _try_load_terraform_outputs() -> dict:
    candidates = [
        Path.cwd().parent / "terraform",
        Path.cwd().parent.parent / "terraform",
    ]
    for tf_dir in candidates:
        if not (tf_dir / ".terraform").exists():
            continue
        try:
            result = subprocess.run(
                ["terraform", f"-chdir={tf_dir}", "output", "-json"],
                check=True,
                capture_output=True,
                text=True,
                timeout=30,
            )
        except Exception:
            continue
        return {k: v.get("value") for k, v in json.loads(result.stdout).items()}
    return {}


# Carrega .env do diretorio do notebook (SageMaker) e da raiz do projeto.
_try_load_dotenv(Path.cwd() / ".env")
_try_load_dotenv(Path.cwd().parent / ".env")

_tf_outputs = _try_load_terraform_outputs()

def _pick(env_key: str, tf_key: str, default: str | None = None) -> str:
    value = os.environ.get(env_key) or _tf_outputs.get(tf_key) or default
    if not value:
        raise RuntimeError(
            f"Configuracao ausente: defina {env_key} ou rode `terraform output` em terraform/."
        )
    return str(value)


AWS_REGION = _pick("AWS_REGION", "aws_region", "us-east-1")
GLUE_DATABASE = _pick("GLUE_DATABASE", "glue_database")
ATHENA_WORKGROUP = _pick("ATHENA_WORKGROUP", "athena_workgroup")
ATHENA_OUTPUT_S3 = _pick("ATHENA_OUTPUT_S3", "athena_output_s3")
DB_SECRET_ID = os.environ.get("DB_SECRET_ID") or _tf_outputs.get("db_secret_id")

boto3.setup_default_session(region_name=AWS_REGION)

print(f"AWS_REGION       = {AWS_REGION}")
print(f"GLUE_DATABASE    = {GLUE_DATABASE}")
print(f"ATHENA_WORKGROUP = {ATHENA_WORKGROUP}")
print(f"ATHENA_OUTPUT_S3 = {ATHENA_OUTPUT_S3}")
print(f"DB_SECRET_ID     = {DB_SECRET_ID or '(nao definido - notebook independe disso para Athena)'}")


In [ ]:
def run_athena(sql: str) -> pd.DataFrame:
    """Helper unico para rodar SQL no Athena dentro do workgroup configurado."""
    return wr.athena.read_sql_query(
        sql=sql,
        database=GLUE_DATABASE,
        workgroup=ATHENA_WORKGROUP,
        ctas_approach=False,
    )


## 4.2 - Consulta exploratoria em dim_products

Inspeciona o catalogo de produtos no modelo analitico.

In [ ]:
df_products = run_athena(
    """
    SELECT
        product_id,
        product_name,
        product_line,
        product_vendor
    FROM dim_products
    LIMIT 20
    """
)

df_products.head(20)


## 4.3 - Vendas totais por pais

Junta fact_orders a dim_countries por country_key e ranqueia os 10 maiores.

In [ ]:
df_sales_country = run_athena(
    """
    SELECT
        dim_countries.country,
        SUM(fact_orders.sales_amount) AS total_sales
    FROM fact_orders
    JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
    GROUP BY dim_countries.country
    ORDER BY total_sales DESC
    LIMIT 10
    """
)

df_sales_country["total_sales"] = df_sales_country["total_sales"].astype(float)
df_sales_country


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=df_sales_country,
    x="total_sales",
    y="country",
    palette="viridis",
    ax=ax,
)
ax.set_title("Top 10 paises por vendas totais")
ax.set_xlabel("Vendas totais (USD)")
ax.set_ylabel("Pais")
plt.tight_layout()
plt.show()


## 4.4 - Detalhamento por data, linha, produto e pais

Combina fact_orders com dim_products, dim_countries e dim_dates no grao de (data, linha de produto, produto, pais).

In [ ]:
df_detail = run_athena(
    """
    SELECT
        dim_dates.full_date,
        dim_products.product_line,
        dim_products.product_name,
        dim_countries.country,
        SUM(fact_orders.sales_amount) AS total_sales
    FROM fact_orders
    JOIN dim_products  ON fact_orders.product_id    = dim_products.product_id
    JOIN dim_countries ON fact_orders.country_key   = dim_countries.country_key
    JOIN dim_dates     ON fact_orders.order_date_key = dim_dates.date_key
    GROUP BY
        dim_dates.full_date,
        dim_products.product_line,
        dim_products.product_name,
        dim_countries.country
    """
)

df_detail["full_date"] = pd.to_datetime(df_detail["full_date"])
df_detail["total_sales"] = df_detail["total_sales"].astype(float)
print(f"linhas: {len(df_detail):,}  |  intervalo: {df_detail['full_date'].min().date()} -> {df_detail['full_date'].max().date()}")
df_detail.head()


## 4.5 - Dashboard interativo

Filtros:

- intervalo de datas sobre full_date;
- pais (com opcao Todos);
- linha de produto (com opcao Todos);
- Top N entre 1 e 10.

O grafico mostra os Top N produtos por vendas totais apos aplicar todos os filtros (agregando sobre o DataFrame ja carregado, sem ir ao Athena a cada interacao).

In [ ]:
_unique_dates = sorted(df_detail["full_date"].dt.date.unique())
_unique_countries = ["Todos"] + sorted(df_detail["country"].dropna().unique().tolist())
_unique_lines = ["Todos"] + sorted(df_detail["product_line"].dropna().unique().tolist())

date_range = SelectionRangeSlider(
    options=[(d.isoformat(), d) for d in _unique_dates],
    index=(0, len(_unique_dates) - 1),
    description="Intervalo:",
    layout={"width": "600px"},
    continuous_update=False,
)
country_dd = Dropdown(options=_unique_countries, value="Todos", description="Pais:")
line_dd = Dropdown(options=_unique_lines, value="Todos", description="Linha:")
top_n_slider = IntSlider(min=1, max=10, value=5, description="Top N:", continuous_update=False)


def render_dashboard(date_range_value, country, product_line, top_n):
    start, end = date_range_value
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)

    filtered = df_detail[
        (df_detail["full_date"] >= start_ts) & (df_detail["full_date"] <= end_ts)
    ]

    if country != "Todos":
        filtered = filtered[filtered["country"] == country]
    if product_line != "Todos":
        filtered = filtered[filtered["product_line"] == product_line]

    if filtered.empty:
        print("Nenhum dado para os filtros selecionados.")
        return

    grouped = (
        filtered.groupby("product_name", as_index=False)["total_sales"].sum()
        .sort_values("total_sales", ascending=False)
        .head(top_n)
    )

    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * top_n + 2)))
    sns.barplot(
        data=grouped,
        x="total_sales",
        y="product_name",
        palette="viridis",
        ax=ax,
    )
    title_country = "todos os paises" if country == "Todos" else country
    title_line = "todas as linhas" if product_line == "Todos" else product_line
    ax.set_title(
        f"Top {top_n} produtos | {title_line} | {title_country} | {start} -> {end}"
    )
    ax.set_xlabel("Vendas totais (USD)")
    ax.set_ylabel("Produto")
    plt.tight_layout()
    plt.show()

    print(f"Linhas pos-filtro: {len(filtered):,}  |  Total vendas: {filtered['total_sales'].sum():,.2f}")


controls = VBox([
    date_range,
    HBox([country_dd, line_dd, top_n_slider]),
])

output = interactive_output(
    render_dashboard,
    {
        "date_range_value": date_range,
        "country": country_dd,
        "product_line": line_dd,
        "top_n": top_n_slider,
    },
)

display(controls, output)


In [ ]:
identity = boto3.client("sts").get_caller_identity()
print({k: identity[k] for k in ("Account", "Arn")})
